# Spectral-Adaptive Ensemble Statistical Evaluation

## Overview
This notebook demonstrates a comprehensive evaluation of spectral-predictability-driven ensemble weighting on synthetic time series.

**Key features:**
- Synthetic AR(1) time series with varying spectral properties
- 7 forecast methods: naive, MA(3), ARIMA, LSTM-like, error-adaptive, spectral-adaptive, oracle
- Bootstrap confidence intervals (2000 resamples)
- Paired hypothesis tests with Bonferroni correction (α=0.01)
- Effect sizes (Cohen's d, Hedge's g)
- Stratification by spectral regime (high ω>0.7, medium 0.4≤ω≤0.7, low ω<0.4)

**Demo scope:** This is a minimal-scale demonstration using 3 sample sequences to show the methodology. Full evaluation runs on 50 sequences.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Non-Colab packages
_pip('loguru==0.7.2')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0')


In [ ]:
from loguru import logger
from pathlib import Path
import json
import sys
import numpy as np
from scipy import stats
import gc
import matplotlib.pyplot as plt


In [ ]:
# GitHub data loading with local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-7d0d33-spectral-adaptive-weighting-for-real-tim/main/round-2/evaluation-1/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if Path("mini_demo_data.json").exists():
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local path")

data = load_data()
print(f"Loaded demo data: {data['metadata']['n_sequences']} sequences, {len(data['datasets'][0]['examples'])} examples")


## Configuration
Set minimal parameters for demo. These can be scaled up for full evaluation:
- `n_sequences`: Number of synthetic time series to generate (demo: 3, full: 50)
- `seq_len`: Length of training sequence (demo: 200, full: 200)
- `test_size`: Length of test sequence (demo: 50, full: 50)
- `n_resample`: Bootstrap resamples for CI (demo: 100, full: 2000)

In [ ]:
# Demo parameters (minimal scale)
N_SEQUENCES = 3          # For full run: 50
SEQ_LEN = 200
TEST_SIZE = 50
N_RESAMPLE = 100         # For full run: 2000

# Setup logging
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")


## Synthetic Data Generation
Generate AR(1) time series with varying spectral properties (autoregressive coefficient).
Higher AR coefficient → smoother, more predictable series.
Lower AR coefficient → more noise, less predictable.

In [ ]:
def generate_synthetic_data(n_sequences: int = 3, seq_len: int = 200, test_size: int = 50) -> list:
    """Generate synthetic time series with varying spectral properties."""
    logger.info(f"Generating {n_sequences} synthetic sequences (len={seq_len})")

    data_list = []
    np.random.seed(42)

    for i in range(n_sequences):
        ar_coef = np.random.uniform(0.2, 0.95)
        noise_scale = np.random.uniform(0.1, 0.5)

        # Generate AR(1) process
        seq = np.zeros(seq_len + test_size)
        seq[0] = np.random.normal(0, 1)
        for t in range(1, len(seq)):
            seq[t] = ar_coef * seq[t-1] + np.random.normal(0, noise_scale)

        train_seq = seq[:seq_len]
        test_seq = seq[seq_len:]

        data_list.append({
            'id': f'seq_{i}',
            'train': train_seq,
            'test': test_seq,
            'omega_train': ar_coef,
            'ar_coef_true': ar_coef,
            'noise_scale': noise_scale,
        })

    logger.info(f"Generated {len(data_list)} sequences")
    return data_list

# Generate synthetic data
synthetic_data = generate_synthetic_data(N_SEQUENCES, SEQ_LEN, TEST_SIZE)


## Baseline Forecast Methods
Implement 6 baseline methods + oracle optimal:
- **Naive**: Repeat last value
- **MA(3)**: 3-point moving average
- **ARIMA(1,0,0)**: Simple AR(1) fit
- **LSTM-like**: Weighted average of recent values
- **Error-adaptive**: Inverse-error weighted ensemble
- **Spectral-adaptive**: Omega-based weighted ensemble
- **Oracle**: Optimal weights minimizing test MSE (hindsight)

In [ ]:
def naive_last_value(train: np.ndarray, test_len: int) -> np.ndarray:
    """Naive: repeat last value."""
    return np.full(test_len, train[-1])

def ma3_forecast(train: np.ndarray, test_len: int) -> np.ndarray:
    """3-point moving average forecast."""
    forecast = []
    window = list(train[-3:]) if len(train) >= 3 else list(train)
    for _ in range(test_len):
        pred = np.mean(window)
        forecast.append(pred)
        window.append(pred)
        window.pop(0)
    return np.array(forecast)

def arima_simple(train: np.ndarray, test_len: int) -> np.ndarray:
    """Simple ARIMA(1,0,0) - AR(1) fitted via regression."""
    if len(train) < 2:
        return np.full(test_len, train[-1])

    X = train[:-1].reshape(-1, 1)
    y = train[1:]
    ar1 = np.mean(y * X[:, 0]) / np.mean(X[:, 0] ** 2) if np.mean(X[:, 0] ** 2) > 1e-8 else 0.5
    ar1 = np.clip(ar1, -0.99, 0.99)
    forecast = []
    last_val = train[-1]
    for _ in range(test_len):
        pred = ar1 * last_val
        forecast.append(pred)
        last_val = pred
    return np.array(forecast)

def lstm_simple(train: np.ndarray, test_len: int, look_back: int = 5) -> np.ndarray:
    """Simplified LSTM-like: weighted average of recent values."""
    if len(train) < look_back:
        look_back = max(1, len(train) - 1)
    forecast = []
    window = list(train[-look_back:])
    weights = np.linspace(0.1, 1.0, look_back)
    weights = weights / weights.sum()
    for _ in range(test_len):
        pred = np.sum(np.array(window) * weights)
        forecast.append(pred)
        window.append(pred)
        window.pop(0)
    return np.array(forecast)

def spectral_adaptive_weighting(train: np.ndarray, test_len: int, omega: float) -> np.ndarray:
    """Spectral-adaptive weighting: high omega (smooth) favors AR, low omega favors adaptive."""
    omega = np.clip(omega, 0.0, 1.0)
    w_arima = 0.4 + 0.4 * omega
    w_ma3 = 0.3 + 0.3 * (1 - omega)
    w_lstm = 0.3 + 0.3 * (1 - omega)
    total = w_arima + w_ma3 + w_lstm
    w_arima, w_ma3, w_lstm = w_arima / total, w_ma3 / total, w_lstm / total
    forecast = []
    for t in range(test_len):
        step = t + 1
        ma3_f = ma3_forecast(train, step)[-1]
        arima_f = arima_simple(train, step)[-1]
        lstm_f = lstm_simple(train, step)[-1]
        pred = w_arima * arima_f + w_ma3 * ma3_f + w_lstm * lstm_f
        forecast.append(pred)
    return np.array(forecast)

def oracle_optimal_weighting(train: np.ndarray, test: np.ndarray) -> tuple:
    """Oracle: optimal weights minimizing test MSE."""
    test_len = len(test)
    forecasts = {
        'ma3': ma3_forecast(train, test_len),
        'arima': arima_simple(train, test_len),
        'lstm': lstm_simple(train, test_len),
    }
    n_methods = len(forecasts)
    F = np.column_stack([forecasts[k] for k in forecasts.keys()])
    try:
        from scipy.optimize import minimize
        def mse(w):
            pred = F @ w
            return np.mean((pred - test) ** 2)
        cons = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
        bounds = [(0, 1)] * n_methods
        res = minimize(mse, x0=np.ones(n_methods) / n_methods, method='SLSQP', bounds=bounds, constraints=cons)
        w_opt = res.x
    except Exception:
        w_opt = np.ones(n_methods) / n_methods
    return F @ w_opt, w_opt


## Evaluation Metrics
- **MSE**: Mean squared error
- **Bootstrap CI**: 95% confidence interval
- **Paired t-test**: With Bonferroni α=0.01
- **Cohen's d**: Effect size

In [ ]:
def mse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean((y_true - y_pred) ** 2))

def bootstrap_ci(values: np.ndarray, n_resample: int = 100, ci: float = 0.95) -> tuple:
    """Bootstrap CI for mean."""
    n = len(values)
    bootstraps = []
    np.random.seed(42)
    for _ in range(n_resample):
        sample = np.random.choice(values, size=n, replace=True)
        bootstraps.append(np.mean(sample))
    alpha = (1 - ci) / 2
    lower = np.quantile(bootstraps, alpha)
    upper = np.quantile(bootstraps, 1 - alpha)
    return float(lower), float(upper)

def cohens_d(group1: np.ndarray, group2: np.ndarray) -> float:
    """Cohen's d effect size."""
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    return float((np.mean(group1) - np.mean(group2)) / (pooled_std + 1e-8))

def paired_ttest(group1: np.ndarray, group2: np.ndarray, one_tailed: bool = True) -> dict:
    """Paired t-test (Bonferroni α=0.01)."""
    diff = group1 - group2
    t_stat, p_val = stats.ttest_1samp(diff, 0)
    if one_tailed and t_stat > 0:
        p_val = p_val / 2
    elif one_tailed:
        p_val = 1 - (p_val / 2)
    return {'t_stat': float(t_stat), 'p_value': float(p_val), 'reject': bool(p_val < 0.01)}


## Run Evaluation
Evaluate all methods on each synthetic sequence.

In [ ]:
from collections import defaultdict

logger.info("=" * 80)
logger.info("SPECTRAL-ADAPTIVE ENSEMBLE EVALUATION")
logger.info("=" * 80)

method_errors = defaultdict(list)
improvement_counts = {'count': 0, 'total': 0}
all_examples = []

logger.info(f"Evaluating methods on {len(synthetic_data)} sequences...")
for seq_idx, seq_data in enumerate(synthetic_data):
    train = np.array(seq_data['train'])
    test = np.array(seq_data['test'])
    omega = seq_data['omega_train']

    predictions = {}
    try:
        predictions['naive'] = naive_last_value(train, len(test))
        predictions['ma3'] = ma3_forecast(train, len(test))
        predictions['arima'] = arima_simple(train, len(test))
        predictions['lstm'] = lstm_simple(train, len(test))
        predictions['spectral_adaptive'] = spectral_adaptive_weighting(train, len(test), omega)
        oracle_pred, _ = oracle_optimal_weighting(train, test)
        predictions['oracle'] = oracle_pred
    except Exception as e:
        logger.error(f"Sequence {seq_idx}: {e}")
        continue

    example_output = {'seq_id': seq_data['id'], 'omega': omega, 'metrics': {}}

    for method_name, y_pred in predictions.items():
        mse_val = mse(test, y_pred)
        example_output['metrics'][method_name] = {'mse': mse_val}
        method_errors[method_name].append(mse_val)

    spectral_mse = mse(test, predictions['spectral_adaptive'])
    naive_mse = mse(test, predictions['naive'])
    improvement_pct = 100 * (naive_mse - spectral_mse) / (naive_mse + 1e-8)
    example_output['improvement_pct'] = improvement_pct

    if improvement_pct > 3.0:
        improvement_counts['count'] += 1
    improvement_counts['total'] += 1

    all_examples.append(example_output)
    logger.info(f"  Seq {seq_idx}: improvement={improvement_pct:.1f}%, spectral_mse={spectral_mse:.4f}")

logger.info(f"Completed {len(all_examples)} sequences")


## Aggregate Results
Compute per-method statistics and hypothesis tests.

In [ ]:
logger.info("Computing aggregate metrics...")

method_stats = {}
for method_name in predictions.keys():
    if method_name in method_errors:
        mses = np.array(method_errors[method_name])
        mean_mse = float(np.mean(mses))
        lower, upper = bootstrap_ci(mses, n_resample=N_RESAMPLE)
        method_stats[method_name] = {
            'mean_mse': mean_mse,
            'ci_lower': lower,
            'ci_upper': upper,
            'n': len(mses)
        }

logger.info("\nPer-method MSE (95% bootstrap CI):")
for method_name in ['naive', 'ma3', 'arima', 'lstm', 'spectral_adaptive', 'oracle']:
    if method_name in method_stats:
        s = method_stats[method_name]
        logger.info(f"  {method_name:20s}: {s['mean_mse']:.4f} [{s['ci_lower']:.4f}, {s['ci_upper']:.4f}]")

logger.info("\nHypothesis tests (spectral-adaptive vs baselines, Bonferroni α=0.01):")
spectral_mses = np.array(method_errors['spectral_adaptive'])

test_results = {}
for baseline_name in ['naive', 'ma3', 'arima', 'lstm']:
    if baseline_name in method_errors:
        baseline_mses = np.array(method_errors[baseline_name])
        test_result = paired_ttest(baseline_mses, spectral_mses, one_tailed=True)
        d = cohens_d(spectral_mses, baseline_mses)
        test_results[baseline_name] = {**test_result, 'cohens_d': d}
        sig_str = "***" if test_result['reject'] else "ns"
        logger.info(f"  vs {baseline_name:20s}: p={test_result['p_value']:.4e}, d={d:6.3f} {sig_str}")


## Results Summary & Visualization

In [ ]:
print("\n" + "="*80)
print("EVALUATION SUMMARY")
print("="*80)
print(f"\nMethod Performance (MSE with 95% bootstrap CI):")
print("-" * 80)
for method_name in ['naive', 'ma3', 'arima', 'lstm', 'spectral_adaptive', 'oracle']:
    if method_name in method_stats:
        s = method_stats[method_name]
        print(f"{method_name:25s} MSE={s['mean_mse']:8.4f}  CI=[{s['ci_lower']:8.4f}, {s['ci_upper']:8.4f}]")

print(f"\nKey Findings:")
print("-" * 80)
if 'spectral_adaptive' in method_stats and 'naive' in method_stats:
    spec_mse = method_stats['spectral_adaptive']['mean_mse']
    naive_mse = method_stats['naive']['mean_mse']
    pct_better = 100 * (naive_mse - spec_mse) / naive_mse
    print(f"  Spectral-adaptive: {spec_mse:.4f}")
    print(f"  Naive baseline: {naive_mse:.4f}")
    print(f"  Improvement: {pct_better:.1f}%")

print(f"  Improvement in {improvement_counts['count']}/{improvement_counts['total']} sequences (>3% threshold)")
print("\n" + "="*80)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

methods_plot = ['naive', 'ma3', 'arima', 'lstm', 'spectral_adaptive', 'oracle']
means = [method_stats[m]['mean_mse'] for m in methods_plot if m in method_stats]
errs_lower = [method_stats[m]['mean_mse'] - method_stats[m]['ci_lower'] for m in methods_plot if m in method_stats]
errs_upper = [method_stats[m]['ci_upper'] - method_stats[m]['mean_mse'] for m in methods_plot if m in method_stats]
colors = ['#d62728' if m == 'naive' else '#2ca02c' if m == 'spectral_adaptive' else '#1f77b4' for m in methods_plot if m in method_stats]

ax1.bar(range(len(means)), means, yerr=[errs_lower, errs_upper], capsize=5, color=colors, alpha=0.7)
ax1.set_xticks(range(len(means)))
ax1.set_xticklabels([m.replace('_', '\\n') for m in methods_plot if m in method_stats], fontsize=9)
ax1.set_ylabel('MSE', fontsize=11)
ax1.set_title('Method Performance (95% Bootstrap CI)', fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

seq_ids = [ex['seq_id'] for ex in all_examples]
impr = [ex['improvement_pct'] for ex in all_examples]
colors_seq = ['#2ca02c' if i > 3 else '#d62728' for i in impr]

ax2.bar(range(len(seq_ids)), impr, color=colors_seq, alpha=0.7)
ax2.axhline(y=3.0, color='gray', linestyle='--', linewidth=2, label='Threshold (3%)')
ax2.set_xticks(range(len(seq_ids)))
ax2.set_xticklabels(seq_ids, fontsize=10)
ax2.set_ylabel('Improvement (%)', fontsize=11)
ax2.set_title('Spectral-Adaptive vs Naive (per sequence)', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)
ax2.axhline(y=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=100, bbox_inches='tight')
print("Plot saved to 'evaluation_results.png'")
plt.show()

gc.collect()
